# Data Science Lab Winter Project 2024/2025
## Age Detection
Within the field of speech processing, a long-standing task of interest is estimating the age of a speaker based on their vocal characteristics. Some systems are capable of analyzing spoken sentences to extract features that correlate with the speaker’s age. The process of estimating a speaker’s age from their speech is referred to as age estimation. The target (output) of the system is a single value representing the estimated age of the speaker.

Report by:
- Gosmar Dario: s337625
- Maretto Chiara: s339401

# 1 Imports
## 1.1 Library Import

In [91]:
# Import the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import librosa

## 1.2 Dataset Import
The dataset is comprised of 3,624 samples: 2,933 samples for the **development set** and 691 samples for the **evaluation set**. Each sample corresponds to a spoken sentence, and the associated speaker’s age is provided as the target label. 
The speech samples have been collected in controlled conditions to ensure consistent feature extraction.

For each sample, a variety of acoustic and linguistic features have been extracted from the speech signal. These features form the dataset and include:
- **sampling rate**: the sampling rate of the audio signal, in Hz
- **age**: the chronological age of the speaker (target label)
- **gender**: the gender of the speake
- **ethnicity**: the ethnicity of the speaker
- **mean pitch**, **max pitch**, **min pitch**: mean, maximum, and minimum pitch of the speech signal, in Hz
- **jitter**: a measure of the variations in pitch, representing voice stability
- **shimmer**: a measure of amplitude variations in the speech signal
- **energy**: the overall energy of the speech signal
- **zcr mean**: the mean zero-crossing rate, indicating the number of times the signal changes sign
- **spectral centroid mean**: the mean spectral centroid, representing the “center of mass” of the frequency spectrum
- **tempo**: the estimated speaking rate, in beats per minute (BPM)
- **hnr**: the harmonic-to-noise ratio, indicating voice quality
- **num words**, **num characters**: the number of words and characters in the spoken sentence
- **num pauses**: the number of pauses detected in the speech
- **silence duration**: the total duration of silence within the speech signal, in seconds
- **path**: the file path to the audio recording

In [92]:
dev_df = pd.read_csv("DSL_Winter_Project_2025/development.csv")
eval_df = pd.read_csv("DSL_Winter_Project_2025/evaluation.csv")

# 2 Data Exploration

In [ ]:
dev_df.info(), dev_df.shape

In [94]:
def get_features(row):
    directory = 'DSL_Winter_Project_2025/'
    file_path = os.path.join(directory, row['path'])
    if os.path.isfile(file_path):
        y, sr = librosa.load(file_path, sr=row['sampling_rate'])
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(np.mean(mfcc, axis=1)) 
        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        spectral_contrast_mean = np.mean(spectral_contrast, axis=1).mean()  
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        spectral_rolloff_mean = np.mean(spectral_rolloff)  

        return librosa.get_duration(y=y, sr=sr) , mfcc_mean , spectral_contrast_mean , spectral_rolloff_mean
    # Return NaNs if the file is missing
    return np.nan, np.nan, np.nan, np.nan 

In [95]:
dev_df[['duration', 'mfcc_mean', 'contrast', 'roloff']] = dev_df.apply(get_features, axis=1, result_type='expand')
eval_df[['duration', 'mfcc_mean', 'contrast', 'roloff']] = eval_df.apply(get_features, axis=1, result_type='expand')

In [96]:
# define a copy of the df to work on
df = dev_df.copy() 

In [97]:
df['tempo'] = df['tempo'].str.replace('[', '').str.replace(']', '').astype('float64')
eval_df['tempo'] = eval_df['tempo'].str.replace('[', '').str.replace(']', '').astype('float64')

In [ ]:
df.describe()

## 2.1 Outliers

In [ ]:
plt.figure(figsize=(6, 2.5))
plt.boxplot(df['age'], vert = False)
plt.title(f'Age Distribution')
plt.ylabel(f'Age')
plt.show()

We can try to remove the outliers of *age*.

In [ ]:
Q1 = dev_df['age'].quantile(0.25)
Q3 = dev_df['age'].quantile(0.87)
IQR = Q3 - Q1

df = df[df['age'] <= (Q3 + 1.5 * IQR)]
df.shape

## 2.2 Correlation between features

In [ ]:
# Plot correlation Matrix
corr = df.drop(columns=['ethnicity','path','gender', 'sampling_rate']).corr(method='spearman')

plt.figure(figsize=(7, 6))
plt.imshow(corr, cmap='magma', interpolation='none', aspect='auto')
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation='vertical')
plt.yticks(range(len(corr)), corr.columns)
plt.suptitle('Feature Correlation Matrix', fontsize=15)
plt.show()

We can see that *num_words* and *num_characters* are very higly correlated, as well as *duration* and *silence_duration*. We decide to remove *num_characters* and *silence_duration* from the dataset.

In [102]:
df = df.drop(columns=['num_characters', 'silence_duration'], axis=1)
eval_df = eval_df.drop(columns=['num_characters', 'silence_duration'], axis=1)

# 3 Data Preprocessing

## 3.1 Numerical features
We can perform normalization on the data do improve in the RMSE

In [103]:
num_feat = df.select_dtypes(include=['float64', 'int64']).columns
num_feat = num_feat.drop('age')
scaler = StandardScaler()
df[num_feat] = scaler.fit_transform(df[num_feat])
eval_df[num_feat] = scaler.transform(eval_df[num_feat])

## 3.2 Categorical features

In [ ]:
dev_df['gender'].unique().size, dev_df['ethnicity'].unique().size

The cardinality of _ethinicity_ is too high to perform 1he, so we perform it on _gender_. Instead, on the ethnicity we can proceed with three possible approaches
- All ethnicities: __eth_group__ = False and __eth__ = True
- Ethnicities grouped: __eth_group__ = True
- No Ethnicities included: __eth_group__ = False and __eth__ = False

By grouping the ethnicities together we aggregate the ethnicity to a higher hierarchical level providing better stability and has less variability. We chose 5 supergroups:
- Indo-European
- Afro-Asiatic
- Sino-Tibetan
- Austroasiatic & Pacific
- Other

Such groups contain ethnicities with similar phonetic characteristics.

In [105]:
eth_group = True
eth = True

We can group ethnicities into broader ethnical categories based on voice and phonetic similarities.

In [ ]:
import squarify

if eth_group:
    # Treemap of the 60 most common ethnicities

    # Calculate the percentage of samples for each ethnicity
    ethnicity_counts = dev_df['ethnicity'].value_counts()
    ethnicity_percentage = (ethnicity_counts / ethnicity_counts.sum()) * 100
    # Prepare data for treemap
    sizes = ethnicity_percentage.values
    labels = ethnicity_percentage.index

    # Plot the treemap
    plt.figure(figsize=(7, 5))
    squarify.plot(sizes=sizes[:60], label=labels[:60], alpha=0.8)
    plt.title('Ethnicity Distribution Treemap')
    plt.axis('off')
    plt.show()
    # Define grouped ethnicities
    ethnicity_groups = {
        "Indo-European": [
            "english", "german", "dutch", "french", "italian", "spanish", "portuguese",
            "russian", "polish", "croatian", "icelandic", "armenian", "hindi", "bengali",
            "bosnian", "azerbaijani", "romanian", "albanian", "macedonian", "irish",
            "greek", "latvian", "lithuanian", "kazakh", "maltese", "finnish",
            "norwegian", "swedish", "danish", "estonian", "luxembourgeois", "faroese",
            "sardinian", "bavarian", "basque"
        ],
        "Afro-Asiatic": [
            "arabic", "hebrew", "amharic", "hausa", "coptic", "amazigh", "kabyle",
            "ethiopian", "akan", "yoruba", "igbo", "edo", "oromo", "kanuri", "fulani",
            "mandinka", "tiv", "bambara", "garifuna", "mende", "lingala", "rundi",
            "swahili", "kikuyu", "luo", "zulu", "xhosa", "bafang", "bamun", "ibibio",
            "mankanya", "ikwerre", "nigerian", "cameroonian", "ijaw", "annang", "fang",
            "ngemba"
        ],
        "Sino-Tibetan": [
            "mandarin", "cantonese", "gan", "hakka", "hainanese", "burmese", "tibetan",
            "chinese"
        ],
        "Austroasiatic & Pacific": [
            "filipino", "indonesian", "malay", "cebuano", "papiamentu", "hmong",
            "khmer", "lao", "vietnamese", "rotuman", "mauritian", "fijian", "pohnpeian", 
            "carolinian", "kiribati", "satawalese","lamotrekese", "mortlockese", "chamorro"
        ],
        "Other": [
            "unknown", "obudu", "ashanti", "kalanga", "nandi", "moba", "sarua",
            "kaire-kaire", "gedeo", "lamaholot", "dinka", "turkish", "kazakh", "kirghiz",
            "finnish", "hungarian", "estonian", "tamil", "telugu", "kannada", "malayalam"
        ]
    }

    # Function to map ethnicities to their corresponding group
    def map_ethnicity(ethnicity):
        for group, ethnicities in ethnicity_groups.items():
            if ethnicity.lower() in ethnicities:
                return group
        return "Other"  # For rare cases not in mapping

    # Apply function to create new column
    df["ethnicity_group"] = df["ethnicity"].apply(map_ethnicity)
    eval_df["ethnicity_group"] = eval_df["ethnicity"].apply(map_ethnicity)

    plt.figure(figsize=(7, 5))
    plt.barh(df['ethnicity_group'].value_counts().index, df['ethnicity_group'].value_counts(), color='skyblue')
    plt.title('Ethnicity Group Distribution')
    plt.xlabel('Number of Samples')
    plt.show()
    encoded_df = pd.get_dummies(df, columns=['gender', 'ethnicity_group'])
    eval_df = pd.get_dummies(eval_df, columns=['gender', 'ethnicity_group'])
    # Ensure eval_df has the same dummy columns as encoded_df
    for col in encoded_df.columns:
        if col not in eval_df.columns:
            eval_df[col] = 0
    eval_df = eval_df[encoded_df.columns].drop(columns=["age"])
    
else:
    if eth:
        encoded_df = pd.get_dummies(df, columns=['gender', 'ethnicity'])
        eval_df = pd.get_dummies(eval_df, columns=['gender', 'ethnicity'])
        # Ensure eval_df has the same dummy columns as encoded_df
        for col in encoded_df.columns:
            if col not in eval_df.columns:
                eval_df[col] = 0
        eval_df = eval_df[encoded_df.columns].drop(columns=["age"])
    else:
        encoded_df = pd.get_dummies(df, columns=['gender'])
        eval_df = pd.get_dummies(eval_df, columns=['gender'])

Now we perform Dummy encoding on the __ethnicity_group__ feature

## 3.3 Feature Selection

We try to perform iterative feature selection using a Random Forest Regressor with default hyperparamters.

In [107]:
df_dropped = encoded_df.drop(columns=['path', 'Id', 'ethnicity']) # drop non-numeric columns 'ethnicity'
feature_names = df_dropped.drop(columns=["age"]).columns
X = df_dropped.drop(columns=["age"])
y = df_dropped["age"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
reg = MLPRegressor(random_state=42)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_valid)
r2 = r2_score(y_valid, y_pred)
rmse = root_mean_squared_error(y_valid, y_pred)
print(f"R^2 Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

We use the Ridge regressor to perform regularization and feature reduction.

In [109]:
from sklearn.linear_model import Ridge
reg = Ridge(random_state=42)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_valid)

In [ ]:
# Use the Ridge model to get feature importances
ndx = np.argsort(np.abs(reg.coef_))
features_df = pd.DataFrame(sorted(zip(feature_names[ndx], np.abs(reg.coef_)[ndx]), key=lambda x: x[1], reverse=False), columns=['Feature', 'Importance'])

plt.figure(figsize=(10, 6))
plt.barh(features_df['Feature'], features_df['Importance'], color='skyblue')
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance')
plt.show()

Try to remove the features with low importance.

In [ ]:
col_drop = features_df[features_df['Importance'] < 0.15]['Feature']
print(f"Columns to drop: {col_drop}")
df_dropped = df_dropped.drop(columns=col_drop) # drop non-numeric columns
feature_names = df_dropped.drop(columns=["age"]).columns

In [ ]:
X = df_dropped.drop(columns=["age"])
y = df_dropped["age"]
X_train, X_valid, y_train, y_valid = train_test_split(X,y, shuffle=True, random_state=42)
reg = MLPRegressor(random_state=42)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_valid)
r2 = r2_score(y_valid, y_pred)
rmse = root_mean_squared_error(y_valid, y_pred)
print(f"R^2 Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

In [113]:
eval_df.set_index('Id', inplace=True)
eval_df = eval_df.drop(columns=['path', 'ethnicity'])
eval_df = eval_df.drop(columns=col_drop)

In [114]:
dev_df = df_dropped.copy()

# 4 Model Selection

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import KFold

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=10, random_state=42),
    "Support Vector Machine": SVR(),
    "Deep Learning": MLPRegressor(random_state=42),
}

X = dev_df.drop(columns=["age"])
y = dev_df["age"]

kf = KFold(n_splits=3, shuffle=True, random_state=42)

for name, model in models.items():
    r2_scores = []
    rmse_scores = []
    for train_index, test_index in kf.split(X):
        X_train, X_valid = X.iloc[train_index], X.iloc[test_index]
        y_train, y_valid = y.iloc[train_index], y.iloc[test_index]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_valid)
        r2 = r2_score(y_valid, y_pred)
        rmse = root_mean_squared_error(y_valid, y_pred)
        r2_scores.append(r2)
        rmse_scores.append(rmse)
    print(f"{name}:")
    print(f"R^2 Score: {np.mean(r2_scores):.4f}")
    print(f"RMSE: {np.mean(rmse_scores):.4f}")
    print()

# 5 Hyperparameter Tuning

We perform grid search on the hyperparameters of:
- Support Vector Machine Regressor
- Random Forest Regressor
- Deep Learning Regressor

In [ ]:
# Grid Search on SVR
config = {
    "C": [1,10,20],
    "epsilon": [0.2, 1,5],
    "kernel": ['linear', 'poly', 'rbf']
}

min_rmse = float('inf')
best_params = {
    "C": None,
    "epsilon": None,
    "kernel": None
}

for c in config["C"]:
    for eps in config["epsilon"]:
        for k in config["kernel"]:
            r2_scores = []
            rmse_scores = []
            for train_index, test_index in kf.split(X):
                X_train, X_valid = X.iloc[train_index], X.iloc[test_index]
                y_train, y_valid = y.iloc[train_index], y.iloc[test_index]
                model = SVR(C=c, epsilon=eps, kernel=k)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_valid)
                r2 = r2_score(y_valid, y_pred)
                rmse = root_mean_squared_error(y_valid, y_pred)
                r2_scores.append(r2)
                rmse_scores.append(rmse)
            if np.mean(rmse_scores) < min_rmse:
                min_rmse = np.mean(rmse_scores)
                best_params["C"] = c
                best_params["epsilon"] = eps
                best_params["kernel"] = k
            print(f"SVR (C={c}, epsilon={eps}, kernel={k}):")
            print(f"R^2 Score: {np.mean(r2_scores):.4f}")
            print(f"RMSE: {np.mean(rmse_scores):.4f}")
            print()
print(f"Best Model: (C = {best_params["C"]}, eps = {best_params["epsilon"]}, Kernel = {best_params['kernel']}) achieved RMSE: {min_rmse:.4f}")

In [ ]:
# Grid Search on Random Forest Regressor

config = {
    "n_estimators": [10, 50, 100, 200],
    "max_depth": [None, 10, 20],
    "max_features": ['sqrt', 'log2', None],
    "criterion": ['squared_error', 'absolute_error', 'friedman_mse', 'poisson']
}

min_rmse = float('inf')
best_params = {
    "n_estimators": None,
    "max_depth": None,
    "max_features": None,
    "criterion": None
}

for n in config["n_estimators"]:
    for d in config["max_depth"]:
        for f in config["max_features"]:
            for c in config["criterion"]:
                r2_scores = []
                rmse_scores = []
                for train_index, test_index in kf.split(X):
                    X_train, X_valid = X.iloc[train_index], X.iloc[test_index]
                    y_train, y_valid = y.iloc[train_index], y.iloc[test_index]
                    model = RandomForestRegressor(n_estimators=n, max_depth=d, max_features=f, criterion=c, random_state=42)
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_valid)
                    r2 = r2_score(y_valid, y_pred)
                    rmse = root_mean_squared_error(y_valid, y_pred)
                    r2_scores.append(r2)
                    rmse_scores.append(rmse)
                if np.mean(rmse_scores) < min_rmse:
                    min_rmse = np.mean(rmse_scores)
                    best_params["n_estimators"] = n
                    best_params["max_depth"] = d
                    best_params["max_features"] = f
                    best_params["criterion"] = c
                print(f"Random Forest (n_estimators={n}, max_depth={d}, max_features={f}, criterion={c}):")
                print(f"R^2 Score: {np.mean(r2_scores):.4f}")
                print(f"RMSE: {np.mean(rmse_scores):.4f}")
                print()
print(f"Best Model: (n_estimators = {best_params['n_estimators']}, max_depth = {best_params['max_depth']}, max_features = {best_params['max_features']}, criterion = {best_params['criterion']}) achieved RMSE: {min_rmse:.4f}")

In [ ]:
# Grid Search on MLPRegressor

config = {
    "hidden_layer_sizes": [(100,), (100, 100), (100, 100, 100)],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate": ['constant', 'adaptive'],
}

min_rmse = float('inf')
best_params = {
    "hidden_layer_sizes": None,
    "alpha": None,
    "learning_rate": None,
}

for h in config["hidden_layer_sizes"]:
    for a in config["alpha"]:
        for l in config["learning_rate"]:
            r2_scores = []
            rmse_scores = []
            for train_index, test_index in kf.split(X):
                X_train, X_valid = X.iloc[train_index], X.iloc[test_index]
                y_train, y_valid = y.iloc[train_index], y.iloc[test_index]
                model = MLPRegressor(hidden_layer_sizes=h, alpha=a, learning_rate=l, random_state=42)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_valid)
                r2 = r2_score(y_valid, y_pred)
                rmse = root_mean_squared_error(y_valid, y_pred)
                r2_scores.append(r2)
                rmse_scores.append(rmse)
            if np.mean(rmse_scores) < min_rmse:
                min_rmse = np.mean(rmse_scores)
                best_params["hidden_layer_sizes"] = h
                best_params["alpha"] = a
                best_params["learning_rate"] = l
            print(f"MLPRegressor (hidden_layer_sizes={h}, alpha={a}, learning_rate={l}):")
            print(f"R^2 Score: {np.mean(r2_scores):.4f}")
            print(f"RMSE: {np.mean(rmse_scores):.4f}")
            print()
print(f"Best Model: (hidden_layer_sizes = {best_params['hidden_layer_sizes']}, alpha = {best_params['alpha']}, learning_rate = {best_params['learning_rate']}) achieved RMSE: {min_rmse:.4f}")

# 6 Evaluation

In [61]:
# We make the predictions using the tuned SVR model

reg = SVR(C=10, epsilon=5, kernel='rbf')
X = dev_df.drop(columns=["age"])
y = dev_df["age"]
reg.fit(X, y)
y_pred = reg.predict(eval_df)

with open('TESTS/predictions_SVR_allout_age.csv', 'w') as f:
    f.write("Id,Predicted\n")
    for i, val in enumerate(y_pred):
        f.write(f"{i},{val}\n")

In [39]:
# We make the predictions using the tuned Random Forest Regressor model

reg = RandomForestRegressor(n_estimators=200, max_depth=10, max_features='sqrt', criterion='friedman_mse', random_state=42)
X = dev_df.drop(columns=["age"])
y = dev_df["age"]
reg.fit(X, y)
y_pred = reg.predict(eval_df)

with open('TESTS/predictions_RandomForest.csv', 'w') as f:
    f.write("Id,Predicted\n")
    for i, val in enumerate(y_pred):
        f.write(f"{i},{val}\n")

In [ ]:
# We make the predictions using the tuned MPLRegressor model

reg = MLPRegressor(hidden_layer_sizes=(100,), alpha=0.001, learning_rate='constant', random_state=42)
X = dev_df.drop(columns=["age"])
y = dev_df["age"]
reg.fit(X, y)
y_pred = reg.predict(eval_df)

with open('TESTS/predictions_MLP_out.csv', 'w') as f:
    f.write("Id,Predicted\n")
    for i, val in enumerate(y_pred):
        f.write(f"{i},{val}\n")

Results obtained on the leaderboard
- SVR TUNED + Mean MFCC : 9.685
- __SVR TUNED: 9.736__
- MLP STANDARD + ETH GROUPS: 9.822
- MLP TUNED: 9.838
- MLP STANDARD + ETH: 9.994
- RANDOM FOREST TUNED: 10.135
- MLP MINMAX SCALER: 10.165
- MLPREGRESSOR BASELINE: 10.228